In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
# =========================
# 1. Load data
# =========================
df = pd.read_csv('data/RawData/telco_customer_data_v2.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'data/RawData/telco_customer_data_v2.csv'

In [ ]:
import pandas as pd
import numpy as np

# =========================
# 2. Basic overview
# =========================
print("Dataset shape:")
print(df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())


# =========================
# 3. Standardize raw text values
# =========================
# Strip spaces from column names
df.columns = df.columns.str.strip()

# Strip spaces inside object columns
object_cols = df.select_dtypes(include="object").columns
for col in object_cols:
    df[col] = df[col].astype(str).str.strip()

# Replace common fake-missing values with np.nan
missing_like_values = ["", " ", "NA", "N/A", "na", "n/a", "null", "NULL", "None", "none", "unknown", "Unknown"]
df.replace(missing_like_values, np.nan, inplace=True)


# =========================
# 4. Missing values report
# =========================
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values(by="missing_count", ascending=False)

print("\nMissing values report:")
print(missing_report)


# =========================
# 5. Duplicate checks
# =========================
full_duplicates = df.duplicated().sum()
customer_id_duplicates = df["customerID"].duplicated().sum() if "customerID" in df.columns else None

print("\nDuplicate rows:", full_duplicates)
print("Duplicate customerID values:", customer_id_duplicates)


# =========================
# 6. Detect columns with suspicious numeric types
# =========================
print("\nChecking numeric columns stored as object...")

for col in df.columns:
    if df[col].dtype == "object":
        converted = pd.to_numeric(df[col], errors="coerce")
        numeric_ratio = converted.notna().mean()

        # If many values can be converted to numeric, the column may have the wrong type
        if numeric_ratio > 0.7:
            print(f"Column '{col}' may be numeric but is stored as object. Convertible ratio: {numeric_ratio:.2%}")


# =========================
# 7. Convert expected numeric columns
# =========================
expected_numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

for col in expected_numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nData types after numeric conversion:")
print(df[expected_numeric_cols].dtypes)


# =========================
# 8. Negative value checks
# =========================
numeric_cols = df.select_dtypes(include=[np.number]).columns

negative_report = {}
for col in numeric_cols:
    negative_count = (df[col] < 0).sum()
    negative_report[col] = negative_count

negative_report_df = pd.DataFrame.from_dict(
    negative_report, orient="index", columns=["negative_count"]
).sort_values(by="negative_count", ascending=False)

print("\nNegative values report:")
print(negative_report_df)


# =========================
# 9. Zero value checks for important columns
# =========================
important_numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

zero_report = {}
for col in important_numeric_cols:
    if col in df.columns:
        zero_count = (df[col] == 0).sum()
        zero_report[col] = zero_count

zero_report_df = pd.DataFrame.from_dict(
    zero_report, orient="index", columns=["zero_count"]
)

print("\nZero values report:")
print(zero_report_df)


# =========================
# 10. Categorical consistency checks
# =========================
print("\nCategorical value inspection:")

for col in object_cols:
    if col in df.columns:
        print(f"\nColumn: {col}")
        print(df[col].value_counts(dropna=False).head(15))


# =========================
# 11. Expected domain checks
# =========================
expected_categories = {
    "gender": {"Male", "Female"},
    "SeniorCitizen": {"0", "1", 0, 1},
    "Partner": {"Yes", "No"},
    "Dependents": {"Yes", "No"},
    "PhoneService": {"Yes", "No"},
    "MultipleLines": {"Yes", "No", "No phone service"},
    "InternetService": {"DSL", "Fiber optic", "No"},
    "OnlineSecurity": {"Yes", "No", "No internet service"},
    "OnlineBackup": {"Yes", "No", "No internet service"},
    "DeviceProtection": {"Yes", "No", "No internet service"},
    "TechSupport": {"Yes", "No", "No internet service"},
    "StreamingTV": {"Yes", "No", "No internet service"},
    "StreamingMovies": {"Yes", "No", "No internet service"},
    "Contract": {"Month-to-month", "One year", "Two year"},
    "PaperlessBilling": {"Yes", "No"},
    "Churn": {"Yes", "No"}
}

print("\nInvalid category checks:")

for col, valid_values in expected_categories.items():
    if col in df.columns:
        actual_values = set(df[col].dropna().unique())
        invalid_values = actual_values - valid_values
        print(f"{col}: {invalid_values if invalid_values else 'No invalid values found'}")


# =========================
# 12. Rows with major issues
# =========================
issue_flags = pd.DataFrame(index=df.index)

issue_flags["missing_any"] = df.isna().any(axis=1)
issue_flags["duplicate_customerID"] = df["customerID"].duplicated(keep=False) if "customerID" in df.columns else False
issue_flags["negative_tenure"] = df["tenure"] < 0 if "tenure" in df.columns else False
issue_flags["negative_monthly"] = df["MonthlyCharges"] < 0 if "MonthlyCharges" in df.columns else False
issue_flags["negative_total"] = df["TotalCharges"] < 0 if "TotalCharges" in df.columns else False

issue_flags["has_issue"] = issue_flags.any(axis=1)

problem_rows = df[issue_flags["has_issue"]].copy()

print("\nNumber of rows with at least one issue:", problem_rows.shape[0])


# =========================
# 13. Final data quality summary table
# =========================
summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "missing_percent": (df.isna().mean() * 100).round(2).values,
    "unique_values": df.nunique(dropna=True).values
})

print("\nFinal data quality summary:")
print(summary.sort_values(by="missing_count", ascending=False))


# =========================
# 14. Optional export
# =========================
print("Data understanding phase completed.")

Dataset shape:
(70000, 21)

Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Data types:
customerID           object
gender               object
SeniorCitizen        object
Partner              object
Dependents           object
tenure              float64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object

Missing values:
cu